# Phase 5 : Robustness Checks

**Project:** A data-driven surrogate for airfoil shape optimization, built on the AirfRANS benchmark

**Author:** Muhammad Uns Haider Shah

> **Status: not yet executed.** This notebook was added in the October 2026 review to test claims that Phases 2–4 rest on but did not check. Run it after re-running notebooks 02, 02b, 03 and 04 (see `RERUN.md`); its results decide which README statements stand.

## Checks
1. **Precision.** Did fp16 inference in the original Phase 2 run affect the field model's drag? Compares the archived fp16 forces with the fp32 re-run.
2. **Seed variation.** How much do the direct surrogates' test metrics move between training seeds? Every number in the README comes from a single run.
3. **Using all 200 cases.** The final direct models were fitted on 160 cases; does refitting on all 200 after model selection change the picture?
4. **About 600 further unseen simulations.** The *full* training cases outside the *scarce* split were never used. Are there other confidently wrong predictions, and are they concentrated in thin sections at negative incidence as Phase 4 suggests?
5. **Separation, measured.** Phase 4 inferred separation in the failure case from a plot. Here reversed near-wall flow is measured directly, for the failure case and for the worst and best predicted unseen cases.
6. **Cross-evaluation of optima.** The Phase 3 optima were found on the ensemble. Here the Gaussian process is optimised too, every optimum is rated by both surrogates, and the disagreement at the optima is compared with the typical disagreement over the design box.

**Runtime:** Google Colab, CPU is sufficient (about 30–40 minutes, most of it computing reference forces for section 6). Results are cached on Drive, so an interrupted run resumes.

## 1. Environment

In [ ]:
# pyvista is pinned: 0.49+ requires a newer IPython than the Colab runtime provides
!pip install -q "airfrans==0.1.5.1" "pyvista==0.48.0" "vtk<9.7"
# Shared helpers live in src/airfrans_surrogate of the project repository: use the local checkout
# when this notebook is opened from one, otherwise clone it (the repository is public).
import os, sys, subprocess
REPO_URL = 'https://github.com/uns-haider96/airfrans-surrogate-optimization'
REPO_DIR = next((d for d in ('..', '.') if os.path.isdir(os.path.join(d, 'src', 'airfrans_surrogate'))), None)
if REPO_DIR is None:
    REPO_DIR = '/content/airfrans-surrogate-optimization'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '-q', REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, os.path.join(os.path.abspath(REPO_DIR), 'src'))
from airfrans_surrogate import print_versions
import airfrans as af
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
import os, json, time, joblib, warnings
from scipy.optimize import minimize
from scipy.stats import qmc, spearmanr
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.exceptions import ConvergenceWarning

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)
print_versions()

## 2. Dataset and earlier artefacts

In [ ]:
from airfrans_surrogate.data import default_paths, prepare_dataset

USE_DRIVE = True        # Colab only; set False to download without caching (required outside Colab)
DRIVE_DIR, ROOT_DL, WORK_DIR = default_paths(USE_DRIVE)
prepare_dataset(USE_DRIVE, DRIVE_DIR, ROOT_DL)

In [ ]:
from airfrans_surrogate.data import locate_root, load_manifest

# The loader needs the folder that contains manifest.json; the local zip is deleted once extracted
ROOT = locate_root(ROOT_DL)
print('Dataset root:', ROOT)
manifest = load_manifest(ROOT)
for k, v in manifest.items():
    print(f'{k:15s} {len(v):5d} simulations')

In [ ]:
from airfrans_surrogate.data import parse_name, family
from airfrans_surrogate.geometry import shape_descriptor, near_wall_reversal
from airfrans_surrogate.models import ForceMLP
from airfrans_surrogate.metrics import comparable_pairs, pair_accuracy

P1_DIR, P2_DIR, P2B_DIR, P3_DIR, P4_DIR = (os.path.join(WORK_DIR, p) for p in ('phase1', 'phase2', 'phase2b', 'phase3', 'phase4'))
OUT_DIR = os.path.join(WORK_DIR, 'phase5')
os.makedirs(OUT_DIR, exist_ok=True)

ens_ck = torch.load(f'{P2B_DIR}/force_ensemble.pt', map_location=DEVICE, weights_only=False)
gp_ck = joblib.load(f'{P2B_DIR}/force_gp.joblib')
CFG2B = ens_ck['cfg']
STATIONS = np.array(ens_ck['stations'])
S = {k: np.array(v) for k, v in ens_ck['stats'].items() if k in ('xm', 'xs', 'ym', 'ys')}
with open(f'{P3_DIR}/phase3_summary.json') as f:
    p3 = json.load(f)
OPT = p3['opt']
optima = np.load(f'{P3_DIR}/optima.npz')
with open(f'{P4_DIR}/phase4_summary.json') as f:
    failure_name = json.load(f)['failure_case']['name']

train_forces = pd.read_csv(f'{P2B_DIR}/train_forces.csv')
test_forces = pd.read_csv(f'{P2_DIR}/test_forces.csv')
with open(f'{P1_DIR}/phase1_summary.json') as f:
    VAL_NAMES = set(json.load(f)['val_names'])
train_only = train_forces[~train_forces['name'].isin(VAL_NAMES)].reset_index(drop=True)
print(f'training {len(train_only)} | validation {len(VAL_NAMES)} | test {len(test_forces)} | '
      f"ensemble width {ens_ck['width']}, weight decay {ens_ck['weight_decay']:g}")

## 3. Shared functions
The same features, training procedure and Gaussian-process kernel as Phase 2b, so that only the factor under study changes.

In [ ]:
features = lambda df: np.c_[np.stack([shape_descriptor(parse_name(n)[2], STATIONS) for n in df['name']]),
                            df['aoa'].values, df['re'].values / 1e6]
targets = lambda df: np.c_[df['true_cl'].values, np.log(df['true_cd'].values)]
nx = lambda X: (X - S['xm']) / S['xs']
ny = lambda Y: (Y - S['ym']) / S['ys']
T = lambda a: torch.as_tensor(a, dtype=torch.float32, device=DEVICE)

def train_ensemble(X, Y, seeds):
    Xt, Yt, nets = T(nx(X)), T(ny(Y)), []
    for seed in seeds:
        torch.manual_seed(seed)
        net = ForceMLP(X.shape[1], ens_ck['width']).to(DEVICE)
        opt = torch.optim.AdamW(net.parameters(), lr=CFG2B['lr'], weight_decay=ens_ck['weight_decay'])
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG2B['epochs'])
        for _ in range(CFG2B['epochs']):
            loss = ((net(Xt) - Yt) ** 2).mean()
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        nets.append(net.eval())
    return nets

@torch.no_grad()
def ens_predict(nets, X):
    P = torch.stack([n(T(nx(X))) for n in nets]).cpu().numpy() * S['ys'] + S['ym']
    return P.mean(0), P.std(0)                      # [CL, log CD] and their spread

def fit_gps(X, Y, seed=0):
    kernel = (ConstantKernel(1.0, (1e-2, 1e2))
              * Matern(length_scale=np.ones(X.shape[1]), length_scale_bounds=(1e-2, 1e3), nu=2.5)
              + WhiteKernel(1e-3, (1e-8, 1e0)))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', ConvergenceWarning)
        return [GaussianProcessRegressor(kernel, normalize_y=False, n_restarts_optimizer=CFG2B['gp_restarts'],
                                         random_state=seed).fit(nx(X), ny(Y)[:, k]) for k in range(2)]

def gp_predict(gps, X):
    out = [g.predict(nx(X), return_std=True) for g in gps]
    return (np.c_[out[0][0], out[1][0]] * S['ys'] + S['ym'],
            np.c_[out[0][1], out[1][1]] * S['ys'])

ensemble = [ForceMLP(len(S['xm']), ens_ck['width']).to(DEVICE) for _ in ens_ck['states']]
for net, st in zip(ensemble, ens_ck['states']):
    net.load_state_dict(st); net.eval()
gps = gp_ck['gps']

EVAL = dict(pair_daoa=1.0, pair_dre=0.5e6, pair_min_rel=0.05)     # as in Phases 2 and 2b
pi, pj, near = comparable_pairs(test_forces['aoa'].values, test_forces['re'].values, EVAL['pair_daoa'], EVAL['pair_dre'])

def test_metrics(cl, cd, sd_logcd, df=test_forces):
    rel = lambda p, t: np.median(np.abs(p - t) / np.abs(t))
    ld, ld_true = cl / cd, df['true_cl'].values / df['true_cd'].values
    z = np.abs(np.log(cd) - np.log(df['true_cd'].values)) / sd_logcd
    return dict(cd_rel_median=rel(cd, df['true_cd'].values), cl_rel_median=rel(cl, df['true_cl'].values),
                rho_d=spearmanr(cd, df['true_cd'])[0], rho_ld=spearmanr(ld, ld_true)[0],
                cd_pairs_distinguishable=pair_accuracy(df['true_cd'].values, cd, pi, pj, near, EVAL['pair_min_rel'])[1],
                logcd_within_2sd=(z <= 2).mean())

## 4. Precision of the field-model forces
The original Phase 2 run evaluated the field models under fp16 autocast; notebook 02 now runs in fp32 and archives the earlier results as `test_forces_fp16.csv`. If the drag columns barely move, the Phase 2 drag diagnosis stands as written.

In [ ]:
fp16_csv = f'{P2_DIR}/test_forces_fp16.csv'
precision = None
if not os.path.exists(fp16_csv) or 'precision' not in test_forces.columns:
    print('Re-run notebook 02 first: it archives the fp16 results and recomputes test_forces.csv in fp32.')
else:
    a = pd.read_csv(fp16_csv).set_index('name')
    b = test_forces.set_index('name').loc[a.index]
    rows = []
    for col in ('sa_ns_cd', 'sa_ns_cdp', 'sa_ns_cdv', 'sa_ns_cl', 'pw_ns_cd', 'pw_ns_cl'):
        ref = b['true_' + col.split('_')[-1]]
        rows.append(dict(quantity=col,
                         fp16_rel_err_median=np.median(np.abs(a[col] - ref) / ref.abs()),
                         fp32_rel_err_median=np.median(np.abs(b[col] - ref) / ref.abs()),
                         fp32_vs_fp16_median_change=np.median(np.abs(b[col] / a[col] - 1)),
                         fp16_spearman=spearmanr(a[col], ref)[0], fp32_spearman=spearmanr(b[col], ref)[0]))
    precision = pd.DataFrame(rows).set_index('quantity')
    print(precision.to_string(float_format=lambda v: f'{v:.4f}'))

## 5. Seed variation and refitting on all 200 cases
Five ensembles with independent seeds, and the Gaussian process with five optimiser seeds, all trained on the same 160 cases; then both refitted on all 200 scarce cases (training + validation) with the selected hyperparameters.

In [ ]:
X_tr, Y_tr = features(train_only), targets(train_only)
X_all, Y_all = features(train_forces), targets(train_forces)
X_te = features(test_forces)
n_mem = CFG2B['n_members']

rows, t0 = [], time.time()
for rep in range(5):
    mu, sd = ens_predict(train_ensemble(X_tr, Y_tr, range(1000 * (rep + 1), 1000 * (rep + 1) + n_mem)), X_te)
    rows.append(dict(model='NN ensemble', fit_on='160', replicate=rep, **test_metrics(mu[:, 0], np.exp(mu[:, 1]), sd[:, 1])))
    mu, sd = gp_predict(fit_gps(X_tr, Y_tr, seed=rep), X_te)
    rows.append(dict(model='Gaussian process', fit_on='160', replicate=rep, **test_metrics(mu[:, 0], np.exp(mu[:, 1]), sd[:, 1])))
    print(f'  replicate {rep + 1}/5 | {time.time() - t0:.0f} s')
mu, sd = ens_predict(train_ensemble(X_all, Y_all, range(n_mem)), X_te)
rows.append(dict(model='NN ensemble', fit_on='200', replicate=0, **test_metrics(mu[:, 0], np.exp(mu[:, 1]), sd[:, 1])))
mu, sd = gp_predict(fit_gps(X_all, Y_all), X_te)
rows.append(dict(model='Gaussian process', fit_on='200', replicate=0, **test_metrics(mu[:, 0], np.exp(mu[:, 1]), sd[:, 1])))

seeds = pd.DataFrame(rows)
seed_table = seeds.drop(columns='replicate').groupby(['model', 'fit_on']).agg(['mean', 'std'])
print(seed_table.T.to_string(float_format=lambda v: f'{v:.4f}'))
print('\nfit_on = 200 is a single fit, so its std is empty.')

## 6. About 600 further unseen simulations
The *full* task's 800 training cases include the 200 *scarce* ones; the other 600, minus any that also appear in the test set, were never used by this project. Their reference forces are computed once and cached. The direct surrogates from Phase 2b (fitted on 160 cases) are then evaluated on them.

In [ ]:
extra_names = sorted(set(manifest['full_train']) - set(manifest['scarce_train']) - set(manifest['full_test']))
CSV = f'{OUT_DIR}/unseen_full_train_forces.csv'
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame()
todo = [n for n in extra_names if not len(done) or n not in set(done['name'])]
print(f'{len(extra_names)} unseen cases; {len(todo)} still to compute')
rows, t0 = [], time.time()
for k, name in enumerate(todo):
    sim = af.Simulation(root=ROOT, name=name)
    u, aoa, _ = parse_name(name)
    (cd, cdp, cdv), (cl, _, _) = sim.force_coefficient(reference=True)
    rows.append(dict(name=name, u_inf=u, aoa=aoa, re=u / float(sim.NU), family=family(name),
                     true_cd=cd, true_cdp=cdp, true_cdv=cdv, true_cl=cl))
    if len(rows) == 25 or k == len(todo) - 1:
        done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); done.to_csv(CSV, index=False); rows = []
        print(f'  {k + 1}/{len(todo)} | {time.time() - t0:.0f} s')
unseen = pd.read_csv(CSV)

X_un = features(unseen)
mu_gp, sd_gp = gp_predict(gps, X_un)
mu_nn, _ = ens_predict(ensemble, X_un)
unseen = unseen.assign(gp_cd=np.exp(mu_gp[:, 1]), gp_sd_logcd=sd_gp[:, 1], nn_cd=np.exp(mu_nn[:, 1]),
                       thickness=[parse_name(n)[2][-1] for n in unseen['name']])
unseen['err'] = np.abs(np.log(unseen['gp_cd']) - np.log(unseen['true_cd']))
unseen['ratio'] = unseen['err'] / unseen['gp_sd_logcd']
unseen['thin_negative'] = (unseen['thickness'] < 8) & (unseen['aoa'] < 0)
unseen['pressure_share'] = unseen['true_cdp'] / unseen['true_cd']

summary_rows = []
for lab, g in (('all unseen', unseen), ('thin (< 8 %) at negative incidence', unseen[unseen['thin_negative']]),
               ('everything else', unseen[~unseen['thin_negative']])):
    summary_rows.append(dict(group=lab, n=len(g), gp_cd_rel_err_median=np.median(np.abs(g['gp_cd'] / g['true_cd'] - 1)),
                             nn_cd_rel_err_median=np.median(np.abs(g['nn_cd'] / g['true_cd'] - 1)),
                             within_2sd=(g['ratio'] <= 2).mean(), ratio_over_5=(g['ratio'] > 5).mean(),
                             ratio_over_20=(g['ratio'] > 20).sum()))
unseen_summary = pd.DataFrame(summary_rows).set_index('group')
print(unseen_summary.to_string(float_format=lambda v: f'{v:.4f}'))
print('\nmost confidently wrong unseen cases (GP drag):')
print(unseen.nlargest(10, 'ratio')[['name', 'aoa', 're', 'thickness', 'true_cd', 'gp_cd', 'pressure_share', 'ratio']]
      .to_string(index=False, float_format=lambda v: f'{v:.4g}'))
print(f"\nSpearman correlation between pressure-drag share and error/sd: "
      f"{spearmanr(unseen['pressure_share'], unseen['ratio'])[0]:.3f}")

## 7. Separation, measured
For each surface node, the velocity at the nearest off-wall node (the first cell) is projected on the surface tangent pointing towards the trailing edge; a negative value means reversed near-wall flow. Nodes ahead of x/c = 0.05 are ignored, since the stretch between the nose and the stagnation point is flagged by construction. Cases: the Phase 4 failure case, the ten most confidently wrong unseen cases, and ten unseen cases predicted within their uncertainty.

In [ ]:
def separation(name):
    sim = af.Simulation(root=ROOT, name=name)
    rev, upper, x = near_wall_reversal(sim.position, sim.normals, sim.sdf, sim.velocity, sim.surface)
    keep = x > 0.05
    frac = lambda side: (rev & side & keep).sum() / max((side & keep).sum(), 1)
    onset = lambda side: x[rev & side & keep].min() if (rev & side & keep).any() else np.nan
    return dict(name=name, reversed_upper=frac(upper), reversed_lower=frac(~upper),
                onset_x_upper=onset(upper), onset_x_lower=onset(~upper))

well = unseen[unseen['ratio'] < 1].sample(10, random_state=0)['name'].tolist()
cases = ([('failure case (Phase 4)', failure_name)] + [('confidently wrong', n) for n in unseen.nlargest(10, 'ratio')['name']]
         + [('within uncertainty', n) for n in well])
sep = pd.DataFrame([dict(group=g, **separation(n)) for g, n in cases])
sep = sep.merge(pd.concat([unseen, train_forces.assign(ratio=np.nan)])[['name', 'aoa', 'ratio']].drop_duplicates('name'),
                on='name', how='left')
print(sep.to_string(index=False, float_format=lambda v: f'{v:.3f}'))
print('\nmean fraction of the surface with reversed near-wall flow, by group:')
print(sep.groupby('group')[['reversed_upper', 'reversed_lower']].mean().round(3).to_string())

## 8. Cross-evaluation of the optima
Problem 1 (maximise L/D at Re = 4 million and 4°) is solved again with the Gaussian-process mean as the objective: a dense Sobol sample, then L-BFGS-B from the best five points. Every optimum is rated by both surrogates. The disagreement at the optima is compared with the typical disagreement over the design box, which shows whether the optimizer is exploiting a surrogate's error.

In [ ]:
lo = np.array([p3['bounds'][k][0] for k in ('M', 'P', 'XX')])
hi = np.array([p3['bounds'][k][1] for k in ('M', 'P', 'XX')])

def design_features(designs, aoa=OPT['aoa_p1']):
    G = np.stack([shape_descriptor(d, STATIONS) for d in np.atleast_2d(designs)])
    return np.c_[G, np.full(len(G), aoa), np.full(len(G), OPT['re'] / 1e6)]

def ld_gp(designs):
    mu, sd = gp_predict(gps, design_features(designs))
    return mu[:, 0] / np.exp(mu[:, 1]), sd[:, 1]

def ld_nn(designs):
    mu, _ = ens_predict(ensemble, design_features(designs))
    return mu[:, 0] / np.exp(mu[:, 1])

box = qmc.scale(qmc.Sobol(3, scramble=True, seed=0).random_base2(15), lo, hi)
ld_box_gp, _ = ld_gp(box)
starts = box[np.argsort(-ld_box_gp)[:5]]
runs = [minimize(lambda x: -ld_gp(x)[0][0], s, method='L-BFGS-B', bounds=list(zip(lo, hi))) for s in starts]
gp_opt = min(runs, key=lambda r: r.fun).x

designs = {'ensemble, gradient-based (Phase 3)': optima['p1_grad'], 'ensemble, Bayesian optimization (Phase 3)': optima['p1_bo'],
           'ensemble, random search (Phase 3)': optima['p1_rand'], 'Gaussian process optimum (here)': gp_opt}
cross = pd.DataFrame([dict(design=k, M=v[0], P=v[1], XX=v[2], LD_ensemble=ld_nn(v)[0], LD_gp=ld_gp(v)[0][0],
                           gp_sd_logcd=ld_gp(v)[1][0]) for k, v in designs.items()]).set_index('design')
cross['disagreement_pct'] = 100 * (cross['LD_ensemble'] / cross['LD_gp'] - 1)
print(cross.to_string(float_format=lambda v: f'{v:.3f}'))

dis_box = 100 * np.abs(ld_nn(box[:4096]) / ld_gp(box[:4096])[0] - 1)
print(f'\nensemble vs GP disagreement over 4096 designs in the box: median {np.median(dis_box):.2f} %, '
      f'90th percentile {np.percentile(dis_box, 90):.2f} %')

## 9. Saved artefacts

In [ ]:
summary = dict(precision=None if precision is None else precision.reset_index().to_dict(orient='records'),
               seeds=seeds.to_dict(orient='records'),
               unseen=unseen_summary.reset_index().to_dict(orient='records'),
               separation=sep.to_dict(orient='records'),
               cross_evaluation=cross.reset_index().to_dict(orient='records'),
               box_disagreement_pct=dict(median=float(np.median(dis_box)), p90=float(np.percentile(dis_box, 90))))
with open(f'{OUT_DIR}/phase5_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)
unseen.to_csv(f'{OUT_DIR}/unseen_full_train_predictions.csv', index=False)
print('saved', f'{OUT_DIR}/phase5_summary.json')

## Summary
*To be written after the first run.* `RERUN.md` lists the README statements each section confirms or revises:

* Section 4: whether the Phase 2 drag figures (1715 % median error, ρ_D = 0.075) change in fp32.
* Section 5: error bars for the Phase 2b table; whether refitting on 200 cases changes it.
* Section 6: whether the "139× the predicted uncertainty" failure is isolated or a pattern, and whether such cases are concentrated in thin sections at negative incidence.
* Section 7: whether the failure case is separated, and whether confidently wrong cases are separated more often than well-predicted ones.
* Section 8: whether the ensemble–GP disagreement at the Phase 3 optimum is larger than the typical disagreement, i.e. whether the optimizer exploited the ensemble's error.